---
# Lesson 9 (optional): ONNX Embedder

> **Note:** Jawad skipped this optional lesson and didn't work on it.
> **Claude added this notebook** (plus `download.py` and `embedder.py`) at Jawad's request,
> following the course's Lesson 9 (`02-vector-search/09-onnx-embedder.md`).

**Simple version:** `sentence-transformers` is like driving a whole truck full of tools (PyTorch, about 4.8 GB)
just to use one screwdriver. **ONNX Runtime** is just the screwdriver (about 147 MB, roughly 33x smaller),
and it gives the **same** vectors. That matters in production: small Docker images, serverless functions, small devices.

**What we do here:**
- Download a ready-made ONNX copy of `all-MiniLM-L6-v2` (the same model as Lessons 2-8).
- Repeat the earlier tests and check that we get the same results, without PyTorch.

**Step 1: download the model (only once).** It saves `model.onnx` and `tokenizer.json` into `models/` (git ignores that folder). Same as running `uv run python download.py` in a terminal.

In [ ]:
from download import download

download("Xenova/all-MiniLM-L6-v2")

**Step 2: load the Embedder.** It works just like `SentenceTransformer`: you call `.encode(text)` and get a vector back.

In [ ]:
from embedder import Embedder

embed = Embedder()

**Step 3: same test as before.** Two questions against one FAQ answer about registering.

In [ ]:
q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

How similar is the "join the course" question to the answer? A bigger number means closer in meaning.

In [ ]:
v1.dot(dv)

And the Docker question? It should be much lower, because it's about something else.

In [ ]:
v2.dot(dv)

**Step 4: embed the whole FAQ.** Load the cards, glue each question to its answer, and turn them into vectors in batches of 50.

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [ ]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

**Step 5: search.** Score every card against the question with a dot product, and take the best one. This is the same numpy search as Lesson 4, just with the lighter embedder.

In [ ]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

**Other models.** The same code works with other ONNX models from HuggingFace (for example
`Xenova/bge-small-en-v1.5`, or `Xenova/multilingual-e5-small` for many languages). Download one with
`download("Xenova/<name>")`, then load it with `Embedder("models/Xenova/<name>")`.

**End of Lesson 9.** Same embeddings, same search results, about 33x lighter to install. Use `sentence-transformers` for experiments, and ONNX Runtime when you ship.

---

---
# Lesson 10: Next steps

**Simple version:** Which search should you use?
- Start with **keyword search**: it's simple and fast.
- Add **vector search** only when you notice people asking in different words than the FAQ uses.
- Best of both: **hybrid search**. Run both searches and merge the results (for example with Reciprocal Rank Fusion: cards ranked high in both lists win).
- **Reranking**: a second "judge" model re-orders the best candidates for even better results.

Ideas to try: compare keyword vs vector search on your own data, try other embedding models, or try another vector database (Qdrant, Elasticsearch, Chroma...).

**End of Lesson 10.** No code here. **Module 2 is complete!**

---